<div style="max-width:900px; margin:40px auto 30px auto; padding:30px 24px;
            text-align:center; font-family:Arial, Helvetica, sans-serif;">

  <div style="font-size:15px; font-weight:600; letter-spacing:0.18em;
              text-transform:uppercase; color:#555; margin-bottom:8px;">
    EPIC Jr II 2026
  </div>

  <div style="font-size:18px; letter-spacing:0.08em;
              color:#777; margin-bottom:26px;">
    Hands-On Astro
  </div>

  <div style="width:70px; height:2px; background:#333;
              margin:0 auto 26px auto;"></div>

  <div style="font-size:42px; font-weight:700; line-height:1.15;
              color:#222; margin-bottom:50px;">
    Dos cúmulos, dos relojes
  </div>

  <div style="font-size:13px; letter-spacing:0.10em;
              text-transform:uppercase; color:#888; margin-bottom:6px;">
    Autores del Proyecto
  </div>

  <div style="font-size:16px; color:#444; line-height:1.7;">
    Pablo Escobar (Yachay Tech) &nbsp;&middot;&nbsp; Ariana Guerrón (USFQ)
  </div>

</div>

<div style="width:72%; margin:55px 0 30px 0; padding:3px 0 14px 18px;
            font-family:Arial, Helvetica, sans-serif;
            border-left:3px solid #526b84;">

  <div style="font-size:15px; font-weight:700; letter-spacing:0.14em;
              text-transform:uppercase; color:#526b84; margin-bottom:7px;">
    Sesión 2
  </div>

  <div style="font-size:30px; font-weight:650; line-height:1.25;
              color:#222; margin-bottom:6px;">
    Corrijamos la distancia y el polvo

  </div>

  <div style="font-size:15px; color:#666; line-height:1.5;">
    ¿ Cómo ponemos ambos cúmulos en una escala física comparable?

  </div>

</div>

## Preparación en Google Colab

1. Abran el notebook en Google Colab.
2. Ejecuten las celdas en orden con el botón **▶**.
3. Cuando se solicite, suban `two_clusters.csv`.

No necesitan instalar paquetes ni consultar catálogos astronómicos.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from IPython.display import display

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.dpi": 120, "axes.titlesize": 14, "axes.labelsize": 12, "font.size": 11,})

CLUSTERS = ["Pleiades", "Messier 67"]
LABELS = {"Pleiades": "Pléyades", "Messier 67": "Messier 67"}
COLORS = {"Pleiades": "#2F80ED", "Messier 67": "#F2994A"}

print("Herramientas preparadas.")

In [ ]:
# Buscar el CSV y, si no está disponible, abrir el cargador de Colab.
candidate_paths = [
    Path("Project_B/student_data/two_clusters.csv"),
    Path("student_data/two_clusters.csv"),
    Path("two_clusters.csv"),
    Path("/content/two_clusters.csv"),
]
data_path = next((path for path in candidate_paths if path.exists()), None)

if data_path is None:
    try:
        from google.colab import files
        print("Suban ahora el archivo two_clusters.csv")
        uploaded = files.upload()
        if "two_clusters.csv" not in uploaded:
            raise FileNotFoundError("No se subió two_clusters.csv.")
        data_path = Path("two_clusters.csv")
    except ImportError as exc:
        raise FileNotFoundError(
            "No se encontró two_clusters.csv. Colóquenlo junto al notebook."
        ) from exc

stars = pd.read_csv(data_path, dtype={"gaia_source_id": "string"})
required_columns = {
    "cluster", "gaia_source_id", "g_mag", "bp_mag", "rp_mag",
    "membership_probability", "use_star", "distance_pc",
    "a_g_mag", "e_bp_rp_mag",
}
missing = required_columns.difference(stars.columns)
if missing:
    raise ValueError(f"Faltan columnas: {sorted(missing)}")
if set(stars["cluster"].dropna()) != set(CLUSTERS):
    raise ValueError("El archivo no contiene exactamente los dos cúmulos esperados.")

print(f" Archivo cargado: {data_path}")
print(f" Filas disponibles: {len(stars):,}")
display(stars.head(3))

In [ ]:
# Recuperación del trabajo científico del día 1.
# Estas dos operaciones ya fueron estudiadas y aquí permiten reiniciar Colab.
stars["bp_minus_rp"] = stars["bp_mag"] - stars["rp_mag"]
selected = stars.loc[stars["use_star"]].copy()

assert len(selected) > 0
print(f"Trabajo del día 1 recuperado: {len(selected)} estrellas seleccionadas.")

<div style="width:100%; margin:5px 0 0px 0; padding:0px 0 0px 0px;
            font-family:Arial, Helvetica, sans-serif;">

  <div style="font-size:18px; font-weight:700; letter-spacing:0.14em;
              text-transform:uppercase; color:#526b84; margin-bottom:7px;">
    4. Corrijan la distancia y el polvo
  </div>
</div>

Antes de calcular, respondan: **¿qué cúmulo necesitará la corrección de distancia mayor y por qué?**

El módulo de distancia es:

$$\mu=5\log_{10}\left(\frac{d}{10\;\mathrm{pc}}\right).$$

Después corregiremos:

$$(G_{\rm BP}-G_{\rm RP})_0=(G_{\rm BP}-G_{\rm RP})-E(G_{\rm BP}-G_{\rm RP}),$$

$$M_{G,0}=G-\mu-A_G.$$

El polvo hace que una estrella parezca más débil y roja. Los valores de corrección ya están preparados para cada cúmulo.

> **Predicción:** escriban aquí el nombre del cúmulo y su razón.

In [ ]:
# TAREA 4: calculen el módulo de distancia para cada fila.
# Usen np.log10, distance_pc y la fórmula anterior.
stars["distance_modulus"] = None

if stars["distance_modulus"].isna().all():
    raise ValueError("Calculen distance_modulus.")
if not np.isfinite(stars["distance_modulus"]).all():
    raise ValueError("El módulo contiene valores no finitos.")

distance_results = (stars.groupby("cluster")
    .agg(distance_pc=("distance_pc", "first"),
        distance_modulus=("distance_modulus", "first"),).reindex(CLUSTERS))
display(distance_results.round(3))

### Comprueben el módulo de distancia

- ¿El cúmulo más lejano obtuvo el módulo mayor?
- ¿Qué representa restar ese valor a una magnitud aparente?

**Checkpoint:** hay un valor finito y diferente para cada cúmulo.

In [ ]:
assert distance_results["distance_modulus"].notna().all()
assert distance_results["distance_modulus"].nunique() == 2
print("Módulos de distancia preparados.")

In [ ]:
# TAREA 5: apliquen las dos correcciones.
# color corregido = color observado − enrojecimiento
# magnitud absoluta = G − módulo de distancia − extinción G
stars["corrected_colour"] = None
stars["absolute_g"] = None

if stars[["corrected_colour", "absolute_g"]].isna().all().any():
    raise ValueError("Completen corrected_colour y absolute_g.")

selected = stars.loc[stars["use_star"]].copy()
assert np.isfinite(selected[["corrected_colour", "absolute_g"]]).all().all()
print("Distancia y polvo corregidos.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.8), sharex=True, sharey=True)

for ax, cluster in zip(axes, CLUSTERS):
    group = selected.loc[selected["cluster"] == cluster]
    ax.scatter(
        group["corrected_colour"], group["absolute_g"],
        s=24, alpha=0.72, color=COLORS[cluster], edgecolor="none",
    )
    ax.set_title(LABELS[cluster])
    ax.set_xlabel(r"Color corregido $(G_{BP}-G_{RP})_0$ (mag)")
    ax.set_xlim(-0.5, 3.6)
    ax.set_ylim(12.8, -1.2)

axes[0].set_ylabel(r"Magnitud absoluta corregida $M_{G,0}$ (mag)")
fig.suptitle("CMD corregidos: misma escala física", fontsize=16, fontweight="bold", y=1.02)
fig.tight_layout()
fig.savefig("day_2_corrected_cmds.png", dpi=220, bbox_inches="tight")
plt.show()
print("Figura guardada como day_2_corrected_cmds.png")

### Lean la comparación y cierren el día

Señalen visualmente:

- la secuencia principal inferior;
- la zona donde la secuencia superior termina o gira;
- estrellas rojas evolucionadas;
- estrellas que no siguen el patrón principal.

> **Evidencia del día 2:** escriban dos diferencias entre las poblaciones sin consultar las edades conocidas.

**Checkpoint final:** los dos paneles tienen límites idénticos; el equipo puede señalar aproximadamente el turn-off de cada cúmulo y explicar qué cambió entre el CMD aparente y el corregido.

Descarguen el notebook editado y `day_2_corrected_cmds.png` antes de salir.